# 40 — Generate the JSON-LD instance contexts

One context per schema, matching decision D1. Outputs at repo root:

- `cosmos_bc_v1.context.jsonld`
- `cosmos_sdtm_v1.context.jsonld`

The 2026-08-30 probe recorded `gen-jsonld-context` as blocked. That was measured
on a schema importing **both** models; per schema it is not blocked, and both
contexts generate. See `docs/decisions.md` D1.

**A context is not identity.** Both contexts map `conceptId` to `@id`, so a
consumer applying one to instance JSON gets a subject IRI only if the value is
already a CURIE or an IRI. The published exports carry bare C-codes, which resolve
against the base and produce nothing usable. `45_identity_probe.ipynb` measures
exactly that; decision D2 says what this repo does about it.

## Configuration

In [ ]:
ROOT      = ".."
DOWNLOADS = "../downloads"
BUILD     = "../build"

SOURCES = {
    "cosmos_bc_v1.context.jsonld":   f"{BUILD}/cosmos_bc_model.patched.yaml",
    "cosmos_sdtm_v1.context.jsonld": f"{DOWNLOADS}/cosmos_sdtm_model.yaml",
}

JSONLD_VERSION = 1.1

## Generate, then post-process

Generated through the Python API rather than the `gen-jsonld-context` CLI. The
two disagree again, as they did for OWL (decision D6) — but this time the API is
the better one: the CLI wraps its output in a `comments` block carrying a
`generation_date`, so the deliverable would differ on every run and a `git diff`
would never mean anything. The API emits `@context` and nothing else.

One authored addition: **`"@version": 1.1`**. The generated context already uses
JSON-LD 1.1 type-scoped contexts for the enum-ranged slots. Declaring it makes a
1.0 processor fail loudly instead of silently misreading them.

The result has exactly one top-level key, `@context` — the same shape as
`usdm-rdf/usdm_v4.context.jsonld`.

In [ ]:
import json
from pathlib import Path

from linkml.generators.jsonldcontextgen import ContextGenerator

for target, source in SOURCES.items():
    document = json.loads(ContextGenerator(source).serialize())

    if sorted(document) != ["@context"]:
        raise RuntimeError(f"{source}: generator produced {sorted(document)}, expected only @context")

    context = {"@version": JSONLD_VERSION, **document["@context"]}

    Path(ROOT, target).write_text(
        json.dumps({"@context": context}, indent=2) + "\n", encoding="utf-8"
    )
    print(f"{target:32s} {len(context) - 1:>3} terms  <- {source}")

## Confirm

Counts are printed. The shape assertions are fail-fast: one top-level key, an
explicit `@version`, and `conceptId` mapped to `@id` — that last one is the hinge
decision D2 turns on, so a change to it should not pass quietly.

In [ ]:
for target in SOURCES:
    document = json.loads(Path(ROOT, target).read_text(encoding="utf-8"))

    if sorted(document) != ["@context"]:
        raise RuntimeError(f"{target}: expected only @context, got {sorted(document)}")

    context = document["@context"]
    if context.get("@version") != JSONLD_VERSION:
        raise RuntimeError(f"{target}: @version is {context.get('@version')!r}")
    if context.get("conceptId") != "@id":
        raise RuntimeError(f"{target}: conceptId maps to {context.get('conceptId')!r}, not @id")

    prefixes = {k: v for k, v in context.items() if isinstance(v, str) and not k.startswith("@")}
    scoped = {k for k, v in context.items() if isinstance(v, dict) and "@context" in v}

    print(f"{target}")
    print(f"    terms            {len(context) - 1}")
    print(f"    prefix bindings  {len(prefixes)}")
    print(f"    type-scoped      {len(scoped)}")
    print(f"    @vocab           {context.get('@vocab')}")

## Provenance

Generated from the models at the commit pinned in `10_fetch_cosmos.ipynb` — the
BC context from the patched copy in `../build/`, so it carries the repaired
namespace (`docs/known-gaps.md` §1a).